In [ ]:

import pandas as pd
import sys
sys.path.insert(0, '..')
from src.features import build_feature_set
from src.forecast import seasonal_naive_forecast, rolling_origin_backtest

df = pd.read_csv('../data/processed/analysis_ready.csv', parse_dates=['date'])

In [ ]:
# Cell 2 - build features + baseline (leakage-safe: every lag/rolling feature is shifted before computing)
df = seasonal_naive_forecast(df)
df = build_feature_set(df)
df.head()

In [ ]:
# Cell 3 - rolling-origin backtest: train on the past, test on the next 28 days, repeat 4 times
results, model = rolling_origin_backtest(df)
results

In [ ]:
# Cell 4 - honest comparison
print(f"Average Model WAPE: {results['model_wape'].mean():.4f}")
print(f"Average Baseline WAPE: {results['baseline_wape'].mean():.4f}")
improvement = (results['baseline_wape'].mean() - results['model_wape'].mean()) / results['baseline_wape'].mean()
print(f"Model improves on baseline by: {improvement:.1%}")

In [ ]:
# Cell 5 - feature importance (which signals actually drove the model)
import pandas as pd
importance = pd.Series(model.feature_importances_, index=model.feature_name_).sort_values(ascending=False)
importance.plot(kind='barh', figsize=(8, 5), title='Feature importance')